# Day 4 · 실습 2. SAM 2.1 Promptable Segmentation

**연계 이론:** Day 4 · 이론 3. Image Segmentation과 SAM  
**권장 운영:** 50분 / 3개 프롬프트의 총 추론 목표 15분 이내(설치·가중치 다운로드 제외)  
**실습 유형:** B(조건 변경·비교) + C(시각화·관찰) / 학습(Training) 없음

### 학습 목표
1. SAM 2.1 Hiera Tiny를 이용하여 단일 이미지에 대한 Mask를 예측합니다.
2. **Positive Point → Positive+Negative Points → Box Prompt**로 입력 조건을 바꿉니다.
3. Prompt별 Mask의 변화와 겹침을 시각화합니다.

> **중요:** 여기서 계산하는 Mask 사이 IoU/Dice는 **서로 다른 프롬프트의 출력 간 일치도**이며, Ground Truth가 없으므로 실제 분할 정확도 점수가 아닙니다.

## 0. 환경과 샘플 이미지 확인

앞선 YOLO 실습과 같은 `requirements.txt`를 사용합니다. SAM 2.1 모델은 최초 한 번 가중치를 다운로드하므로 담당자가 수업 전에 네트워크 또는 `weights/sam2.1_t.pt` 파일을 준비해야 합니다.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ZIP 안의 assets 폴더를 기준으로 작업 경로를 찾습니다.
def locate_root():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        for candidate in (parent, parent / "cv_day4_practice_v1"):
            if (candidate / "assets" / "coffee.jpg").is_file():
                return candidate
    raise FileNotFoundError(
        "assets/coffee.jpg를 찾지 못했습니다. ZIP 전체를 압축 해제하고 "
        "cv_day4_practice_v1 폴더에서 Notebook을 실행하세요."
    )

ROOT = locate_root()
ASSET_DIR = ROOT / "assets"
OUTPUT_DIR = ROOT / "outputs"
WEIGHT_DIR = ROOT / "weights"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("실습 루트:", ROOT)
print("샘플 이미지:", [p.name for p in sorted(ASSET_DIR.glob("*.jpg"))])

In [ ]:
import torch
try:
    import ultralytics
    from ultralytics import SAM
except ImportError as exc:
    raise ImportError("ultralytics 미설치. %pip install -r requirements.txt 실행 후 커널 재시작이 필요합니다.") from exc

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)
print("장치:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (SAM 처리 시간이 길 수 있음)")

SAM_MODEL_NAME = "sam2.1_t.pt"  # SAM 2.1 Hiera Tiny / 경량 모델
MODEL_SOURCE = str(WEIGHT_DIR / SAM_MODEL_NAME) if (WEIGHT_DIR / SAM_MODEL_NAME).is_file() else SAM_MODEL_NAME
try:
    sam = SAM(MODEL_SOURCE)
except Exception as exc:
    raise RuntimeError("SAM 2.1 모델을 로드하지 못했습니다. 모델 가중치/인터넷 연결을 확인하세요.") from exc
print("모델:", MODEL_SOURCE)

## 1. 프롬프트 위치와 의미 살펴보기

`coffee.jpg`의 크기는 600×400입니다. 아래 좌표는 **픽셀 좌표 (x, y)**이며, 단일 커피잔을 분할하는 예시입니다. 프롬프트를 다르게 지정하면 선택되는 영역이 달라질 수 있습니다. 정확한 대상은 실행 결과를 보면서 판단합니다.

In [ ]:
IMAGE_NAME = "coffee.jpg"
IMAGE_PATH = ASSET_DIR / IMAGE_NAME
image_rgb = np.asarray(Image.open(IMAGE_PATH).convert("RGB"))
H, W = image_rgb.shape[:2]
print("이미지 Shape (H, W, C):", image_rgb.shape)

# [조건 변경] Point 및 Box 좌표를 바꾸어 비교할 수 있습니다.
POSITIVE = [250, 210]  # 컵 표면의 한 지점
NEGATIVE = [460, 310]  # 주변 접시/배경의 한 지점
BOX = [165, 15, 425, 315]  # 컵을 둘러싼 XYXY Box

assert all(0 <= p[0] < W and 0 <= p[1] < H for p in [POSITIVE, NEGATIVE])
assert 0 <= BOX[0] < BOX[2] <= W and 0 <= BOX[1] < BOX[3] <= H

from matplotlib.patches import Rectangle
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(image_rgb)
ax.scatter(*POSITIVE, c="lime", s=130, marker="o", edgecolors="black", label="Positive (+)")
ax.scatter(*NEGATIVE, c="red", s=130, marker="x", linewidths=3, label="Negative (-)")
ax.add_patch(Rectangle((BOX[0], BOX[1]), BOX[2]-BOX[0], BOX[3]-BOX[1], fill=False, edgecolor="yellow", linewidth=2, label="Box"))
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.legend(loc="upper right")
ax.set_title("Prompt locations — edit the coordinate variables above")
plt.show()

## 2. 세 가지 프롬프트로 SAM 2.1 분할

Ultralytics SAM2.1 공식 API의 `points`, `labels`, `bboxes`를 사용합니다. `labels=1`은 Positive, `labels=0`은 Negative입니다. 여러 점을 **하나의 객체에 대한 프롬프트**로 전달하기 위해 배열을 `[[[x,y], ...]]`처럼 묶습니다.

In [ ]:
import cv2

SAM_OUT = OUTPUT_DIR / "sam"
SAM_OUT.mkdir(parents=True, exist_ok=True)

def first_mask(results, size_hw):
    """Ultralytics Results에서 첫 번째 Mask를 (H,W) bool로 변환합니다."""
    if not results or results[0].masks is None or len(results[0].masks.data) == 0:
        raise RuntimeError("Mask 예측이 없습니다. 프롬프트 좌표를 실제 대상 내부로 조정하세요.")
    arr = results[0].masks.data[0].detach().float().cpu().numpy()
    if arr.shape != tuple(size_hw):
        arr = cv2.resize(arr, (size_hw[1], size_hw[0]), interpolation=cv2.INTER_NEAREST)
    return (arr > 0.5).astype(bool)

# 동일한 이미지·모델·장치를 유지하고 Prompt만 변경합니다.
PROMPTS = {
    "positive": {"points": [[POSITIVE]], "labels": [[1]]},
    "positive_negative": {"points": [[POSITIVE, NEGATIVE]], "labels": [[1, 0]]},
    "box": {"bboxes": BOX},
}

masks = {}
for label, prompt in PROMPTS.items():
    try:
        result = sam.predict(
            source=str(IMAGE_PATH), device=DEVICE, save=False, verbose=False,
            **prompt
        )
    except Exception as exc:
        raise RuntimeError(f"SAM 실행 실패 ({label}). 프롬프트 형식, GPU 메모리, 모델 버전을 확인하세요.") from exc
    masks[label] = first_mask(result, (H, W))
    np.save(SAM_OUT / f"mask_{label}.npy", masks[label])
    print(f"{label:>18} | Mask 픽셀 수: {int(masks[label].sum()):,}")

## 3. Prompt별 Mask 결과 시각화

다음 그림에서 초록색은 분할된 픽셀입니다. Positive/Negative Point 및 Box를 함께 표시하므로 어떤 입력이 어떤 결과와 연결되는지 확인할 수 있습니다.

In [ ]:
def overlay_mask(img_rgb, mask, opacity=0.44):
    img = img_rgb.astype(np.float32).copy()
    color = np.array([35, 205, 100], dtype=np.float32)
    img[mask] = img[mask] * (1-opacity) + color * opacity
    return np.clip(img, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, key in zip(axes, masks):
    overlay = overlay_mask(image_rgb, masks[key])
    ax.imshow(overlay)
    if key in ("positive", "positive_negative"):
        ax.scatter(*POSITIVE, c="lime", edgecolors="black", s=90)
    if key == "positive_negative":
        ax.scatter(*NEGATIVE, c="red", s=105, marker="x", linewidths=3)
    if key == "box":
        ax.add_patch(Rectangle((BOX[0], BOX[1]), BOX[2]-BOX[0], BOX[3]-BOX[1],
                               fill=False, edgecolor="yellow", linewidth=2))
    ax.set_title(f"{key}\narea={int(masks[key].sum()):,} pixels")
    ax.axis("off")
    Image.fromarray(overlay).save(SAM_OUT / f"overlay_{key}.png")
fig.tight_layout()
fig.savefig(SAM_OUT / "prompt_comparison.png", dpi=140, bbox_inches="tight")
plt.show()

## 4. Mask 변경 정도를 정량적으로 살펴보기

**Prompt A와 B의 일치도**만 비교합니다. 정답 Mask가 없는 상태이므로 이 표를 분할 정확도(성능)로 해석하지 않습니다. IoU와 Dice의 분모에 두 영역의 합집합/면적 합이 들어간다는 점을 다시 확인하세요.

In [ ]:
def compare_masks(a, b):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    intersection = int(np.logical_and(a, b).sum())
    union = int(np.logical_or(a, b).sum())
    total = int(a.sum() + b.sum())
    iou = intersection / union if union else 1.0  # 둘 다 비어 있으면 일치 처리(본 실습 정의)
    dice = 2 * intersection / total if total else 1.0
    return {"intersection": intersection, "union": union, "iou": iou, "dice": dice,
            "a_only": int(np.logical_and(a, ~b).sum()),
            "b_only": int(np.logical_and(b, ~a).sum())}

pairs = [("positive", "positive_negative"), ("positive", "box"), ("positive_negative", "box")]
for a_name, b_name in pairs:
    m = compare_masks(masks[a_name], masks[b_name])
    print(f"{a_name:>18} vs {b_name:<18} | IoU={m['iou']:.3f}, Dice={m['dice']:.3f}, "
          f"A만={m['a_only']:,}, B만={m['b_only']:,}")

# Prompt 사이 영역 차이를 그림으로 표현합니다.
a, b = masks["positive"], masks["positive_negative"]
diff = np.zeros((H, W, 3), dtype=np.uint8)
diff[np.logical_and(a, b)] = [80, 180, 80]     # 공통 영역 - 녹색
diff[np.logical_and(a, ~b)] = [250, 120, 80]  # Positive에만 포함 - 주황
diff[np.logical_and(~a, b)] = [65, 140, 245]  # Positive+Negative에만 포함 - 파랑
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(image_rgb); axes[0].axis("off"); axes[0].set_title("Original image")
axes[1].imshow(diff); axes[1].axis("off")
axes[1].set_title("Green: both / Orange: only + / Blue: only +,-")
fig.tight_layout(); fig.savefig(SAM_OUT / "prompt_difference.png", dpi=140, bbox_inches="tight"); plt.show()

In [ ]:
metadata = {
    "model": SAM_MODEL_NAME,
    "image": IMAGE_NAME,
    "width": W, "height": H,
    "positive": POSITIVE, "negative": NEGATIVE, "box": BOX,
    "mask_pixels": {k: int(v.sum()) for k, v in masks.items()},
    "mask_files": {k: f"mask_{k}.npy" for k in masks},
    "note": "No ground truth: pairwise mask IoU/Dice is agreement, not accuracy."
}
with (SAM_OUT / "sam_prompts.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("저장:", SAM_OUT)
print("다음 Notebook에서 Mask 배열과 비교 이미지를 재사용합니다.")

## 5. 핵심 질문 및 다음 실습 연결

1. Positive Point만 사용했을 때 모델이 어느 영역을 선택했나요?
2. Negative Point를 추가했을 때 어떤 픽셀이 제외/추가되었나요?
3. Box Prompt는 Point Prompt와 어떻게 다른 정보를 제공하나요?
4. Prompt별 두 Mask의 IoU는 **정답과의 일치도**인가요, **두 예측의 일치도**인가요?
5. 실제 Mask IoU·Dice를 평가하려면 무엇이 필요하나요?

**다음:** `day4_04_cv_error_analysis_v1.ipynb`에서는 저장된 YOLO 탐지와 SAM Mask를 재사용하여 오류를 해석합니다.